# Notebook 33 — MLP capacity sweep: a third architecture on the capacity axis

Notebook 32 rejected the capacity hypothesis for the transformer: at matched or lower remaining capacity
than the CNN's prune80, it degrades gracefully (macro-F1 loss 0.070-0.096) where the CNN collapses
(0.271). This notebook places the MLP on the same axis using the five existing MLP baselines
(hidden 256/128, seeds 0-4, trained on the frozen primary split), pruned to 50 / 80 / 90 / 95% with the
same layer-wise pruner and fine-tuning protocol. The head is held at 80% (as for the CNN and the
transformer); the body carries the sweep. 90% leaves roughly the CNN prune80 remainder.

**Gate (stated before running).** As in Notebook 32: H-cap is supported for the MLP if, at the cell
closest to the CNN prune80 remainder, mean paired macro-F1 loss exceeds 0.15 and at least four classes
are materially affected in three or more seeds. The archived single-anchor MLP prune80 result becomes a
five-seed paired result here. Resumable per (seed, sparsity). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET, ARCH = 'ciciot2023', 'mlp'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
AMOUNTS = [(0.50, 'prune50'), (0.80, 'prune80'), (0.90, 'prune90'), (0.95, 'prune95')]

ARCH_KW = {'hidden': (256, 128)}   # the archived MLP anchor configuration (validation gate, Notebook 10)
BASE_CELL = 'M0'                    # MLP baselines seeds 0-4 were saved under the plain M0 cell
print('ARCH_KW:', ARCH_KW)

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Architecture gate: this architecture's VALIDATION macro-F1 (5 seeds) against the OTHER archived bands
gate = pd.read_csv(OUT / 'validation_architecture_gate.csv')
bands = {a: (g.min(), g.max()) for a, g in gate[gate.status == 'ok'].groupby('arch')['validation_macro_f1']}
rows = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    rows.append({'arch': ARCH, 'seed': seed, 'validation_macro_f1': f1_score(yv, pv, average='macro')})
tg = pd.DataFrame(rows); lo, hi = tg.validation_macro_f1.min(), tg.validation_macro_f1.max()
others = {a_: b for a_, b in bands.items() if a_ != ARCH}
overlap = any(not (hi < b[0] or lo > b[1]) for b in others.values())
tg['cnn_band'] = str(tuple(round(x, 3) for x in bands['cnn1d'])); tg['mlp_band'] = str(tuple(round(x, 3) for x in bands['mlp']))
tg['overlaps_archived_bands'] = overlap
tg.to_csv(OUT / 'mlp_validation_gate.csv', index=False)
print(tg.round(4).to_string(index=False))
print(f'\n{ARCH} validation band {lo:.3f}-{hi:.3f} | cnn {bands["cnn1d"][0]:.3f}-{bands["cnn1d"][1]:.3f} | mlp {bands["mlp"][0]:.3f}-{bands["mlp"][1]:.3f}')
print(f'overlaps other archived bands {sorted(others)} (co-equal-arm criterion):', overlap)

In [ ]:
# Pruner (identical to Notebook 31): attention projections included, layer-wise L1, masked fine-tune
def prunable(model):
    out = []
    for mod in model.modules():
        if isinstance(mod, (nn.Linear, nn.Conv1d)): out.append((mod, 'weight'))
        elif isinstance(mod, nn.MultiheadAttention) and getattr(mod, 'in_proj_weight', None) is not None: out.append((mod, 'in_proj_weight'))
    return out
EDGE_AMOUNT = 0.80   # tokenizer and head: held at the archived prune80 level, as in the CNN's prune80
def is_edge(model, mod):
    return mod is getattr(model, 'tokenizer', None) or mod is getattr(model, 'head', None)
def body(model): return [(mod, n) for mod, n in prunable(model) if not is_edge(model, mod)]
def magnitude_prune_ext(model, amount):
    # extreme sparsity on the body only; tiny edge layers would otherwise be destroyed trivially
    m = copy.deepcopy(model)
    for mod, name in prunable(m):
        a = EDGE_AMOUNT if is_edge(m, mod) else amount
        prune.l1_unstructured(mod, name=name, amount=a); prune.remove(mod, name)
    return m
def sparsity_report(model):
    z = n = 0
    for mod, name in prunable(model):
        w = getattr(mod, name); z += int((w == 0).sum()); n += w.numel()
    zb = nb_ = 0
    for mod, name in body(model):
        w = getattr(mod, name); zb += int((w == 0).sum()); nb_ += w.numel()
    tot = sum(p.numel() for p in model.parameters()); ztot = sum(int((p == 0).sum()) for p in model.parameters())
    return {'prunable_sparsity': z / n, 'body_sparsity': zb / nb_, 'whole_model_sparsity': ztot / tot, 'prunable_params': n,
            'body_params': nb_, 'total_params': tot, 'remaining_nonzero_prunable': n - z, 'remaining_nonzero_total': tot - ztot}
def prune_and_finetune_ext(anchor, seed, amount, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = magnitude_prune_ext(anchor, amount).to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    {amount:.3f} ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    rep = sparsity_report(model); assert abs(rep['body_sparsity'] - amount) < 0.02, rep
    return model.eval(), le, scaler, rep
def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)

# CNN reference remainder on the same axis (prunable = Linear + Conv1d weights, as in the archived pruner)
cnn, _, _, _ = load_anchor(DATASET, 'cnn1d', 'M0', ANCHOR, arch_kwargs={'channels': (64, 128)})
cnn_prunable = sum(m.weight.numel() for m in cnn.modules() if isinstance(m, (nn.Linear, nn.Conv1d)))
CNN_P80_REMAINING = int(round(cnn_prunable * 0.2))
print(f'CNN prunable weights {cnn_prunable:,} -> prune80 remainder {CNN_P80_REMAINING:,} nonzero')
_probe = M.build(ARCH, len(feat_cols), int(df.label.nunique()), **ARCH_KW)
_body_n = sum(getattr(m_, n_).numel() for m_, n_ in body(_probe)); _edge_n = sum(getattr(m_, n_).numel() for m_, n_ in prunable(_probe)) - _body_n
for a, c in AMOUNTS: print(f'  mlp {c}: ~{int(_body_n * (1 - a) + _edge_n * (1 - EDGE_AMOUNT)):,} remaining prunable nonzero (body {_body_n:,} @ {a:.3f}, edges {_edge_n:,} @ 0.80)')

In [ ]:
# Sweep with resume: 5 baselines x 4 sparsities
baseline_val, baseline_test, comp_test, macro, sp_rows, pruned = {}, {}, {}, [], [], {}
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
    baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'cell': 'M0', 'test_macro_f1': f1_score(yt, pt, average='macro')})
    for amount, cell in AMOUNTS:
        p_c = PATHS.model(DATASET, ARCH, f'{cell}_mlp_paired', seed)
        if os.path.exists(p_c):
            mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
            mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval()
            rep = sparsity_report(mp); print(f'  loaded {cell}')
        else:
            mp, _, _, rep = prune_and_finetune_ext(m0, seed, amount, verbose=True); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell}')
        sp_rows.append({'seed': seed, 'cell': cell, **rep})
        yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        comp_test[(seed, cell)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
        macro.append({'seed': seed, 'cell': cell, 'test_macro_f1': f1_score(yt, pc, average='macro')})
        pruned[(seed, cell)] = mp
print('\nsweep complete')

In [ ]:
print('comp_test:', len(comp_test), '| macro:', len(macro), '| sparsity rows:', len(sp_rows))

In [ ]:
# Aggregate (Notebook 11/31 schema) + the combined capacity axis
tiers = assign_validation_tiers(pd.DataFrame(baseline_val))
rows = []
for (seed, cell), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'cell': cell, 'class': cls, 'M0_test_recall': float(r0.loc[cls]), 'compressed_test_recall': float(rc.loc[cls]),
                     'recall_loss': loss, 'validation_tier': tiers.loc[cls, 'validation_tier'], 'validation_2sd_band': band,
                     'crosses_validation_band': bool(loss > band), 'practically_material': bool(loss >= PRACTICAL_LOSS),
                     'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); eff.to_csv(OUT / 'mlp_capacity_sweep_per_class_effects.csv', index=False)
summ = eff.groupby(['cell', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), sd_recall_loss=('recall_loss', 'std'),
        affected_frequency=('material_and_beyond_band', 'mean'), n=('seed', 'nunique')).reset_index()
summ.to_csv(OUT / 'mlp_capacity_sweep_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'mlp_capacity_sweep_macro_f1_wide.csv', index=False)
msum = mdf.groupby('cell')['test_macro_f1'].agg(['count', 'mean', 'std', 'min', 'max']).reset_index()
msum.columns = ['cell', 'n', 'mean', 'sd', 'min', 'max']; msum.to_csv(OUT / 'mlp_capacity_sweep_macro_f1_summary.csv', index=False)
spdf = pd.DataFrame(sp_rows); spdf.to_csv(OUT / 'mlp_capacity_sweep_sparsity.csv', index=False)

m0_mean = float(msum.loc[msum.cell == 'M0', 'mean'].iloc[0])
axis = []
# CNN rows from archived paired-seed results
cm = pd.read_csv(OUT / 'paired_seed_macro_f1_summary.csv').set_index('cell')['mean']
cs = pd.read_csv(OUT / 'paired_seed_per_class_summary.csv')
for cell, frac in (('prune50', 0.5), ('prune80', 0.2)):
    axis.append({'arch': 'cnn1d', 'cell': cell, 'remaining_nonzero_prunable': int(round(cnn_prunable * frac)),
                 'mean_macro_f1': float(cm[cell]), 'mean_macro_f1_loss': float(cm['M0'] - cm[cell]),
                 'classes_affected_ge3of5': int((cs[cs.cell == cell].affected_frequency >= 0.6).sum())})
# transformer rows carried over from the committed combined axis (Notebooks 31/32)
prev = pd.read_csv(OUT / 'capacity_axis_combined.csv')
for _, r in prev[prev.arch == 'ft_transformer'].iterrows(): axis.append(r.to_dict())
for _, cell in AMOUNTS:
    rem = int(round(spdf[spdf.cell == cell].remaining_nonzero_prunable.mean()))
    axis.append({'arch': ARCH, 'cell': cell, 'remaining_nonzero_prunable': rem, 'mean_macro_f1': float(msum.loc[msum.cell == cell, 'mean'].iloc[0]),
                 'mean_macro_f1_loss': m0_mean - float(msum.loc[msum.cell == cell, 'mean'].iloc[0]),
                 'classes_affected_ge3of5': int((summ[summ.cell == cell].affected_frequency >= 0.6).sum())})
axis = pd.DataFrame(axis).sort_values(['arch', 'remaining_nonzero_prunable'], ascending=[True, False])
axis.to_csv(OUT / 'capacity_axis_combined.csv', index=False)  # now cnn1d + ft_transformer + mlp
print(msum.round(4).to_string(index=False)); print()
print(axis.round(4).to_string(index=False))

In [ ]:
# Mechanism at the matched-capacity cell (anchor seed): leakage-safe probes + dense-head refit
tf_cells = axis[axis.arch == ARCH]
matched = tf_cells.iloc[(tf_cells.remaining_nonzero_prunable - CNN_P80_REMAINING).abs().argsort().iloc[0]]['cell']
print('matched-capacity cell:', matched, f'(CNN prune80 remainder {CNN_P80_REMAINING:,})')
seed = ANCHOR
m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
pm = pruned[(seed, matched)]
collapsed = list(summ[(summ.cell == matched) & (summ.affected_frequency >= 0.6)]['class'])
print(f'{len(collapsed)} classes materially affected in >=3/5 seeds at {matched}')
pdf = pd.DataFrame(columns=['label', 'auc_M0', 'auc_compressed', 'auc_drop'])
if collapsed:
    fit = {}
    for which in ('train', 'val'):
        f0, _, y = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which=which)
        fc, _, _ = EXP.extract_features(pm, df, splits, scaler, feat_cols, le, which=which); fit[which] = (f0, fc, y)
    f0_te, _, y_te = EXP.extract_features(m0, df, splits, scaler, feat_cols, le, which='test')
    fc_te, _, _ = EXP.extract_features(pm, df, splits, scaler, feat_cols, le, which='test')
    f0_fit = np.concatenate([fit['train'][0], fit['val'][0]]); fc_fit = np.concatenate([fit['train'][1], fit['val'][1]]); y_fit = np.concatenate([fit['train'][2], fit['val'][2]])
    rng = np.random.default_rng(seed); keep = []
    for c in np.unique(y_fit):
        idx = np.where(y_fit == c)[0]; keep.append(rng.choice(idx, 8000, replace=False) if len(idx) > 8000 else idx)
    keep = np.concatenate(keep); f0_fit, fc_fit, y_fit = f0_fit[keep], fc_fit[keep], y_fit[keep]
    def probe(f_fit, f_te, c):
        yb, yt_ = (y_fit == c).astype(int), (y_te == c).astype(int)
        if yb.sum() < 5 or yt_.sum() < 2: return np.nan
        return roc_auc_score(yt_, LogisticRegression(max_iter=500, C=1.0, class_weight='balanced').fit(f_fit, yb).predict_proba(f_te)[:, 1])
    prow = []
    for cname in collapsed:
        c = int(np.where(le.classes_ == cname)[0][0]); a0, ac = probe(f0_fit, f0_te, c), probe(fc_fit, fc_te, c)
        prow.append({'label': cname, 'auc_M0': a0, 'auc_compressed': ac, 'auc_drop': a0 - ac})
    pdf = pd.DataFrame(prow); print(pdf.round(4).to_string(index=False))
pdf.to_csv(OUT / f'mlp_{matched}_leakage_safe_probe.csv', index=False)

Lval, yval, Ltest, ytest = mitigate.refit_head(pm, df, splits, scaler, feat_cols, le, epochs=15, lr=1e-2, batch_size=4096, seed=seed)
refit_f1 = f1_score(ytest, np.asarray(Ltest).argmax(1), average='macro')
cal = calibration_summary(torch.softmax(torch.tensor(Ltest), dim=1).numpy(), ytest); cal.insert(0, 'cell', f'mlp_{matched}_head_refit')
cal.to_csv(OUT / f'mlp_{matched}_head_refit_calibration.csv', index=False)
print(f'\nhead refit macro-F1 at {matched}: {refit_f1:.4f}')

In [ ]:
# Gate verdict - written whichever way it falls
row = axis[(axis.arch == ARCH) & (axis.cell == matched)].iloc[0]
loss = float(row.mean_macro_f1_loss); n_aff = int(row.classes_affected_ge3of5)
min_auc = float(pdf.auc_compressed.min()) if len(pdf) else float('nan')
recovery = (refit_f1 - float(row.mean_macro_f1)) / loss if loss > 0 else float('nan')
any_cell_collapses = bool(((axis.arch == ARCH) & (axis.mean_macro_f1_loss > 0.15) & (axis.classes_affected_ge3of5 >= 4)).any())
verdict = pd.DataFrame([
 {'criterion': 'matched_cell', 'value': matched, 'pass': ''},
 {'criterion': 'matched_mean_macro_f1_loss_gt_0.15', 'value': round(loss, 4), 'pass': loss > 0.15},
 {'criterion': 'matched_classes_affected_ge3of5_ge_4', 'value': n_aff, 'pass': n_aff >= 4},
 {'criterion': 'matched_min_probe_auc_collapsed', 'value': round(min_auc, 4), 'pass': bool(min_auc >= 0.85) if len(pdf) else 'n/a'},
 {'criterion': 'matched_refit_recovery_fraction', 'value': round(recovery, 4), 'pass': bool(recovery >= 0.5) if loss > 0 else 'n/a'},
 {'criterion': 'any_mlp_cell_collapses_up_to_95', 'value': any_cell_collapses, 'pass': any_cell_collapses},
])
h_cap = bool(loss > 0.15 and n_aff >= 4)
print(verdict.to_string(index=False))
print('\nH-cap (capacity governs collapse) supported at matched capacity:', h_cap)
print('H-arch (locality prior governs collapse) supported:', (not h_cap) and (not any_cell_collapses))
verdict.to_csv(OUT / 'mlp_capacity_gate_verdict.csv', index=False)
write_json(OUT / 'mlp_capacity_sweep_environment.json', {'arch_kwargs': ARCH_KW, 'amounts': AMOUNTS, 'cnn_p80_remaining': CNN_P80_REMAINING,
                                                                'matched_cell': matched, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/33_mlp_capacity_sweep.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/mlp_capacity_*') + glob.glob('results/tables/comnet/mlp_prune*')
               + glob.glob('results/tables/comnet/mlp_validation_gate.csv') + glob.glob('results/tables/comnet/capacity_axis_combined.csv'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 33: MLP capacity sweep (50/80/90/95) on five baselines, third architecture on the combined capacity axis, matched-cell mechanism, gate verdict'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)